# NBA SQLite Ad Hoc Explorer

Use this notebook to inspect the schema and run manual SQL queries against `nba.sqlite`.

Assumes the database file lives in the repo root as `nba.sqlite`.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("../nba.sqlite")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

def run_sql(sql: str, params=None):
    """Run a SQL query and return a pandas DataFrame."""
    if params is None:
        params = {}
    return pd.read_sql_query(sql, conn, params=params)

def list_tables():
    return run_sql(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """
    )

def table_info(table_name: str):
    return run_sql(f"PRAGMA table_info('{table_name}')")

def row_count(table_name: str):
    return run_sql(f'SELECT COUNT(*) AS rows FROM "{table_name}"')

print(f"Connected to {DB_PATH.resolve()}")

In [ ]:
# Quick schema snapshot
tables = list_tables()
tables

In [ ]:
# Table summary with row counts
summary = []
for table_name in tables['table_name']:
    summary.append({
        'table': table_name,
        'rows': int(row_count(table_name).iloc[0, 0]),
        'columns': len(table_info(table_name)),
    })

pd.DataFrame(summary).sort_values('rows', ascending=False)

In [ ]:
# Inspect one table in detail
table_name = "game"
table_info(table_name)

In [ ]:
# Run an ad hoc query here.
# Replace the SQL with whatever you want to validate.
sql = """
SELECT COUNT(*) AS total_games
FROM game
"""

run_sql(sql)

In [ ]:
# Example: inspect the most recent games.
run_sql(
    """
    SELECT game_id, game_date, season_id, season_type, matchup_home, wl_home, matchup_away, wl_away
    FROM game
    ORDER BY game_date DESC, game_id DESC
    LIMIT 10
    """
)

When you are done, you can close the connection with `conn.close()`.